In [1]:
!pip install langchain langchain_core langchain_community langchain_huggingface pydantic duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.0 MB/s eta 0:00:00


## Built-in DuckDuckGo Search

In [2]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 2.8 MB/s eta 0:00:00


In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

result = search_tool.invoke('Top 10 business news in NewYorkTimes today')

result

"Find the latest stock market news from every corner of the globe at Reuters.com, your online source for breaking international market and finance news Find the latest business news with reports on Wall Street, interest rates, banking, companies, and U.S. and world financial markets. Subscribe to the Business Story of the Day podcast. Stock Market News And Analysis The analysis you'll find in the Stock Market Today is based on over 130 years of market history and a detailed study of every top-performing stock since the 1880s. CNBC is the world leader in business news and real-time financial market coverage. Find fast, actionable information. The New York Times Company is a publicly traded media organization best known for publishing The New York Times newspaper and operating the NYTimes.com digital platform. The company produces daily print and digital journalism covering national and international news, opinion pieces, feature stories, and multimedia content. Alongside its flagship ne

## Shell Command Tool

In [4]:
!pip install langchain-experimental

In [5]:
from langchain_community.tools import ShellTool
shell_tool = ShellTool()

result = shell_tool.invoke('ls')

result

Executing command:
 ls


/usr/local/lib/python3.12/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


'sample_data\n'

## Custom Tools

In [6]:
from langchain.tools import tool

In [7]:
# Step 1: Create a function
def multiply(a,b):
  """Multiplies two numbers"""
  return a*b

In [8]:
# Step 2: add type hints
def multiply(a:int, b:int) -> int:
  """Multiplies two number"""
  return a*b

In [9]:
# Step 3: add tool decorator
@tool
def multiply(a:int, b:int) -> int:
  """Multiplies two number"""
  return a*b

In [10]:
result = multiply.invoke({'a':3, 'b':5})

In [11]:
result

15

In [12]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiplies two number
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [13]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiplies two number', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## Using StructuredTool

In [14]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [18]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="The number to add")
  b: int = Field(required=True, description = "The second number to add")

/tmp/ipykernel_2389/4210008219.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The number to add")
/tmp/ipykernel_2389/4210008219.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description = "The second number to add")


In [19]:
def multiply_func(a:int, b:int) -> int:
  return a * b

In [20]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description='Multiply two numbers',
    args_schema=MultiplyInput
)

In [21]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Using BaseTool Class

In [22]:
from langchain.tools import BaseTool
from typing import Type

In [23]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="The first number to add")
  b: int = Field(required=True, description="The second number to add")


/tmp/ipykernel_2389/2507307695.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_2389/2507307695.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [25]:
# Using this technique we can apply async logic here also

class MultiplyTool(BaseTool):
  name: str = 'multiply'
  description: str = "Multiply two numbers"

  args_schema: Type[BaseModel] = MultiplyInput

  def _run(self, a: int, b:int) -> int:
    return a * b

In [26]:
multiply_tool = MultiplyTool()

In [27]:
result = multiply_tool.invoke({'a':3,'b':4})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

12
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Toolkit

In [29]:
@tool
def add(a:int,b:int) -> int:
  """Add two numbers"""
  return a+b

@tool
def multiply(a:int,b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [30]:
class MathToolkit:
  def get_tools(self):
    return [add, multiply]

In [31]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
  print(tool.name, "=>" ,tool.description)

add => Add two numbers
multiply => Multiply two numbers
